In [ ]:
from Modules.visual_analyser import VisualAnalyzer
from Modules.data_preprocessing import DataPreprocessing
from Modules.image_analyzer import ImageAnalyzer
import pandas as pd
import os
import random
import numpy as np
import networkx as nx
import igraph as ig

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
# recupération des données
dp = DataPreprocessing()
source_path = ""
df_path = os.path
df = dp.read_data(os.path.join(source_path, "post_rehydrated.pickle"), format_="pickle")
dp.parse_dates()

In [ ]:
df["account_followers"]

In [ ]:
## Nombre de tweets
infos_tweets_cp = {
    "Tweets spreaders": len(df),
    "Nb comptes spreaders": len(df["pf_account_id"].unique()),
}

infos_tweets_cp

In [ ]:
import matplotlib.pyplot as plt


def plot_publications_activity(period=None, df=None):

    df = df.copy()
    df["post_created_at"] = pd.to_datetime(df["post_created_at"], utc=True)

    # Filtrer période si nécessaire
    if period:
        start_date, end_date = period
        df = df[
            (df["post_created_at"] >= pd.to_datetime(start_date, utc=True))
            & (df["post_created_at"] <= pd.to_datetime(end_date, utc=True))
        ]

    # évolution hebdomadaire
    df["week"] = df["post_created_at"].dt.to_period("W").dt.start_time

    weekly_posts = (
        df.groupby(["week", "join_post_post_type"])
        .agg(n_posts=("post_id", "count"))
        .reset_index()
    )

    pivot_posts = weekly_posts.pivot(
        index="week", columns="join_post_post_type", values="n_posts"
    ).fillna(0)

    # métriques globales
    metrics = {
        "Engagements": df["post_engagements"].sum(),
        "Vues": df["post_views"].sum(),
        "Abonnés": df["account_followers"].max(),
        "Suivis": df["account_following"].max(),
    }

    # plots
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # évolution des posts
    for col in pivot_posts.columns:
        axes[0].plot(pivot_posts.index, pivot_posts[col], marker="o", label=col)

    axes[0].set_title("Évolution hebdomadaire des publications par type", fontsize=14)
    axes[0].set_xlabel("Semaine", fontsize=14)
    axes[0].set_ylabel("Nombre de publications", fontsize=14)
    axes[0].legend(title="Type de post")
    axes[0].grid(True)
    axes[0].tick_params("both", labelsize=12)

    # barplot métriques
    axes[1].grid(True)
    axes[1].bar(metrics.keys(), metrics.values())
    axes[1].set_title("Métriques globales", fontsize=14)
    axes[1].set_ylabel("Total", fontsize=14)
    axes[1].tick_params("both", labelsize=12)

    plt.tight_layout()
    plt.show()

In [ ]:
df_temp = df[df["post_created_at"] >= pd.to_datetime("2025-01-01", utc=True)]
plot_publications_activity(df=df_temp)

In [ ]:
print(2.25 * 10e8 / 43779)
print(2.65 * 10e8 / 217580)

On a donc 217 580 tweets des comptes recepteurs. Les comptes uniques sont au nombre de 43 779. Pour un ensemble de 26 148 images. Parmi celles-ci, 5731 images ont été identifiée comme visuellement dupliquées, indiquant une réutilisation fréquente de certains visuels au sein de la plateforme.

Après extraction des caractéristiques visuelles (vectorisation des images) et reduction de dimension via \gls{sUMAP}, nous obtenons 105 clusters d'images, de taille très homogènes (maximum = xxx, minimum = xxx, moyenne = xxx et ecart-type = xxx). Certains clusters rassemblent un grands nombres d'images quasi-identiques, tandis que d'autres regroupent des conenus plus variés. Ce qui reflète des dynamiques de diffusion distinctes. La ditribution temporelle des publications met en évidence plusieurs pics d'activité (commentaires, citations et post-originaux), notamment entre mi-juin et mi-juillet, période durant laquelle les comptes impliqués cumulent en moyenne 51 394 abonnés et génèrent environ 12 179 vues.
Ces observations suggèrent des phases d’intensification de la diffusion du récit, potentiellement associées à des comportements de coordination ou d’amplification non organiques.
Cet echantillon offre ainsi un terrain d'analyse pour étudier à la fois la diffusion organique et les eventuels schémas de coordination dans la propagation des images liées à ***nicolas qui paie***.

In [ ]:
df["join_post_post_type"].unique()